# 05 - Análise Integrada

Este notebook integra as bases consolidadas do SIH/SUS, CNES e população para construção das análises hospitalares do projeto.

## 1. Configuração inicial

In [1]:
from pathlib import Path
import pandas as pd
pasta_silver = Path("../data/silver")

## 2. Carregamento das bases consolidadas

In [2]:
df_sih = pd.read_parquet( pasta_silver / "sih_multianual.parquet")
df_cnes = pd.read_parquet( pasta_silver / "cnes_multianual.parquet")
df_populacao = pd.read_parquet( pasta_silver / "populacao_multianual.parquet")

print("SIH:", df_sih.shape)
print("CNES:", df_cnes.shape)
print("População:", df_populacao.shape)

SIH: (2286755, 19)
CNES: (198448, 12)
População: (22281, 10)


## 3. Verificação das bases carregadas

In [3]:
print("SIH")
display(df_sih.head())

print("CNES")
display(df_cnes.head())

print("População")
display(df_populacao.head())

SIH


,ANO_CMPT,MES_CMPT,N_AIH,IDENT,SEQ_AIH5,CNES,MUNIC_RES,MUNIC_MOV,DT_INTER,DT_SAIDA,DIAS_PERM,UTI_MES_TO,VAL_TOT,DIAG_PRINC,MORTE,ESPEC,PROC_REA,CAR_INT,ARQUIVO_ORIGEM
0,2021,01,5220103703310,1,000,6665322,521930,521930,2020-10-29,20201101,3,0,485.78,K929,0,03,0303070102,02,RDGO2101.dbc
1,2021,01,5220103703320,1,000,6665322,521930,521930,2020-10-31,20201103,3,0,242.88,R31,0,03,0303150050,02,RDGO2101.dbc
2,2021,01,5220103703353,1,000,6665322,521930,521930,2020-10-08,20201009,1,0,40.38,I743,0,03,0301060070,02,RDGO2101.dbc
3,2021,01,5220103703397,1,000,6665322,521300,521930,2020-10-28,20201102,5,0,503.85,C819,0,03,0304100021,02,RDGO2101.dbc
4,2021,01,5220103703419,1,000,6665322,521930,521930,2020-11-11,20201112,1,0,40.38,K922,1,03,0301060070,02,RDGO2101.dbc


CNES


,CNES,CODUFMUN,TP_UNID,TP_LEITO,CODLEITO,QT_EXIST,QT_SUS,QT_NSUS,COMPETEN,ARQUIVO_ORIGEM,ANO,MES
0,9331603,520010,15,2,33,9,9,0,202101,LTGO2101.dbc,2021,1
1,2335506,520013,05,6,34,4,3,1,202101,LTGO2101.dbc,2021,1
2,2335506,520013,05,1,03,2,1,1,202101,LTGO2101.dbc,2021,1
3,2335506,520013,05,5,45,3,3,0,202101,LTGO2101.dbc,2021,1
4,2335506,520013,05,4,43,3,3,0,202101,LTGO2101.dbc,2021,1


População


,uf,cod_uf,cod_municipio,municipio,populacao,codigo_ibge_7,ano_referencia,ano_publicacao,origem,tipo_dado
0,RO,11,00015,Alta Floresta D'Oeste,22516,1100015,2021,2021,IBGE,Estimativa populacional
1,RO,11,00023,Ariquemes,111148,1100023,2021,2021,IBGE,Estimativa populacional
2,RO,11,00031,Cabixi,5067,1100031,2021,2021,IBGE,Estimativa populacional
3,RO,11,00049,Cacoal,86416,1100049,2021,2021,IBGE,Estimativa populacional
4,RO,11,00056,Cerejeiras,16088,1100056,2021,2021,IBGE,Estimativa populacional


In [4]:
print("Período SIH:")
print( df_sih["ANO_CMPT"].min(), df_sih["ANO_CMPT"].max())

print("Período CNES:")
print( df_cnes["ANO"].min(), df_cnes["ANO"].max())

print("Anos de população:")
print(
    sorted(
        df_populacao["ano_referencia"]
        .unique()
    )
)

Período SIH:
2021 2026
Período CNES:
2021 2026
Anos de população:
[np.int64(2021), np.int64(2022), np.int64(2024), np.int64(2025)]


## 4. Compatibilização dos códigos municipais

Os códigos municipais das fontes possuem formatos diferentes. Antes da integração, é construída e validada uma correspondência entre o código IBGE de sete dígitos e os códigos municipais utilizados pelo DATASUS.

In [5]:
df_populacao["codigo_sus_6"] = (
    df_populacao["codigo_ibge_7"]
    .astype("string")
    .str[:6]
)

In [6]:
dim_municipios = (
    df_populacao[
        [
            "codigo_ibge_7",
            "codigo_sus_6",
            "uf",
            "municipio",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_municipios.head()

,codigo_ibge_7,codigo_sus_6,uf,municipio
0,1100015,110001,RO,Alta Floresta D'Oeste
1,1100023,110002,RO,Ariquemes
2,1100031,110003,RO,Cabixi
3,1100049,110004,RO,Cacoal
4,1100056,110005,RO,Cerejeiras


In [7]:
duplicados_codigo_sus = (
    dim_municipios
    .groupby("codigo_sus_6")
    ["codigo_ibge_7"]
    .nunique()
)

duplicados_codigo_sus = duplicados_codigo_sus[
    duplicados_codigo_sus > 1
]

print(
    "Códigos SUS associados a mais de um código IBGE:",
    len(duplicados_codigo_sus)
)

Códigos SUS associados a mais de um código IBGE: 0


## 5. Validação dos códigos municipais com o CNES

In [8]:
municipios_cnes = set(
    df_cnes["CODUFMUN"]
    .astype("string")
    .str.strip()
    .unique()
)

municipios_dim = set(
    dim_municipios["codigo_sus_6"]
    .dropna()
)

cnes_sem_correspondencia = sorted(
    municipios_cnes
    - municipios_dim
)

print(
    "Municípios do CNES sem correspondência:",
    len(cnes_sem_correspondencia)
)

print(
    cnes_sem_correspondencia[:20]
)

Municípios do CNES sem correspondência: 0
[]
